# DSPy SQL Agent(Pokedex) — Versão Aprimorada
> **Equipe:** Cauã  · Davi  · João Paulo  · Kauan  - 3°ADS

Este notebook implementa um agente de consultas SQL em linguagem natural usando **DSPy**,
com signatures avançadas, módulos robustos com *MultiChainComparison*, otimizadores
`BootstrapFewShot` + `MIPROv2`, cache de queries, histórico de execução e pipeline de avaliação completo.


## 1. Setup do Ambiente

In [6]:
# Instala dependências do sistema e Ollama
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Instala DSPy e dependências Python
!pip install -q dspy-ai colorama tabulate


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.1 MB/s eta 0:00:00

In [7]:
#Teste do servidor
# !ollama serve &
# !sleep 3
# !ollama list

In [8]:
import threading
import subprocess
import time

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama_serve, daemon=True)
thread.start()
time.sleep(2)  # aguarda Ollama iniciar
print("✓ Ollama service iniciado em background")


✓ Ollama service iniciado em background


In [9]:
# Baixa o modelo
!ollama pull qwen2.5-coder:3b


## 2. Imports e Configuração Global

In [10]:
import dspy
import sqlite3
import requests
import json
import hashlib
import time
import logging
from datetime import datetime
from typing import Optional
from tabulate import tabulate

# Logging legível
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("dspy_agent")


In [11]:
def setup_ollama(model: str = "qwen2.5-coder:3b", temperature: float = 0.1) -> bool:
    """Verifica se o Ollama está rodando e configura o LM no DSPy."""
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        resp.raise_for_status()
        logger.info("Ollama está rodando ✓")
    except Exception as e:
        logger.error(f"Ollama não acessível: {e}")
        return False

    lm = dspy.LM(
        f"ollama_chat/{model}",
        api_base="http://localhost:11434",
        temperature=temperature,
        max_tokens=512,
        cache=False,          # DSPy cache interno desativado; usamos cache próprio
    )
    dspy.configure(lm=lm)
    logger.info(f"DSPy configurado com modelo: {model}")
    return True

setup_ollama()


True

## 3. Banco de Dados de Exemplo

In [12]:
def create_sample_database(db_path: str = "pokedex.db") -> None:
    """Cria e popula a Pokédex da 1ª geração."""
    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.executescript("""
        CREATE TABLE IF NOT EXISTS pokemon (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL,
            type1    TEXT NOT NULL,
            type2    TEXT,           -- NULL se tiver só um tipo
            hp       INTEGER NOT NULL,
            attack   INTEGER NOT NULL,
            defense  INTEGER NOT NULL,
            sp_atk   INTEGER NOT NULL,
            sp_def   INTEGER NOT NULL,
            speed    INTEGER NOT NULL,
            total    INTEGER NOT NULL
        );

        CREATE TABLE IF NOT EXISTS types (
            id   INTEGER PRIMARY KEY,
            name TEXT NOT NULL UNIQUE
        );

        CREATE TABLE IF NOT EXISTS evolutions (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            from_id      INTEGER NOT NULL,
            to_id        INTEGER NOT NULL,
            min_level    INTEGER,        -- NULL se não evoluir por level
            method       TEXT NOT NULL,  -- 'level', 'stone', 'trade'
            FOREIGN KEY (from_id) REFERENCES pokemon(id),
            FOREIGN KEY (to_id)   REFERENCES pokemon(id)
        );

        CREATE TABLE IF NOT EXISTS moves (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL,
            type     TEXT NOT NULL,
            power    INTEGER,   -- NULL para moves de status
            accuracy INTEGER,
            pp       INTEGER NOT NULL,
            category TEXT NOT NULL  -- 'Physical', 'Special', 'Status'
        );

        CREATE TABLE IF NOT EXISTS pokemon_moves (
            pokemon_id INTEGER NOT NULL,
            move_id    INTEGER NOT NULL,
            learn_method TEXT NOT NULL,  -- 'level-up', 'tm', 'hm'
            learn_level  INTEGER,        -- NULL se não for level-up
            PRIMARY KEY (pokemon_id, move_id, learn_method),
            FOREIGN KEY (pokemon_id) REFERENCES pokemon(id),
            FOREIGN KEY (move_id)    REFERENCES moves(id)
        );
    """)

    # ── Pokémon (todos os 151 da Gen 1) ─────────────────────────────────────
    pokemon_data = [
        (1,  'Bulbasaur',   'Grass',  'Poison', 45,49,49,65,65,45,318),
        (2,  'Ivysaur',     'Grass',  'Poison', 60,62,63,80,80,60,405),
        (3,  'Venusaur',    'Grass',  'Poison', 80,82,83,100,100,80,525),
        (4,  'Charmander',  'Fire',   None,     39,52,43,60,50,65,309),
        (5,  'Charmeleon',  'Fire',   None,     58,64,58,80,65,80,405),
        (6,  'Charizard',   'Fire',   'Flying', 78,84,78,109,85,100,534),
        (7,  'Squirtle',    'Water',  None,     44,48,65,50,64,43,314),
        (8,  'Wartortle',   'Water',  None,     59,63,80,65,80,58,405),
        (9,  'Blastoise',   'Water',  None,     79,83,100,85,105,78,530),
        (10, 'Caterpie',    'Bug',    None,     45,30,35,20,20,45,195),
        (11, 'Metapod',     'Bug',    None,     50,20,55,25,25,30,205),
        (12, 'Butterfree',  'Bug',    'Flying', 60,45,50,90,80,70,395),
        (13, 'Weedle',      'Bug',    'Poison', 40,35,30,20,20,50,195),
        (14, 'Kakuna',      'Bug',    'Poison', 45,25,50,25,25,35,205),
        (15, 'Beedrill',    'Bug',    'Poison', 65,90,40,45,80,75,395),
        (16, 'Pidgey',      'Normal', 'Flying', 40,45,40,35,35,56,251),
        (17, 'Pidgeotto',   'Normal', 'Flying', 63,60,55,50,50,71,349),
        (18, 'Pidgeot',     'Normal', 'Flying', 83,80,75,70,70,101,479),
        (19, 'Rattata',     'Normal', None,     30,56,35,25,35,72,253),
        (20, 'Raticate',    'Normal', None,     55,81,60,50,70,97,413),
        (21, 'Spearow',     'Normal', 'Flying', 40,60,30,31,31,70,262),
        (22, 'Fearow',      'Normal', 'Flying', 65,90,65,61,61,100,442),
        (23, 'Ekans',       'Poison', None,     35,60,44,40,54,55,288),
        (24, 'Arbok',       'Poison', None,     60,95,69,65,79,80,448),
        (25, 'Pikachu',     'Electric',None,    35,55,40,50,50,90,320),
        (26, 'Raichu',      'Electric',None,    60,90,55,90,80,110,485),
        (27, 'Sandshrew',   'Ground', None,     50,75,85,20,30,40,300),
        (28, 'Sandslash',   'Ground', None,     75,100,110,45,55,65,450),
        (29, 'Nidoran♀',   'Poison', None,     55,47,52,40,40,41,275),
        (30, 'Nidorina',    'Poison', None,     70,62,67,55,55,56,365),
        (31, 'Nidoqueen',   'Poison', 'Ground', 90,92,87,75,85,76,505),
        (32, 'Nidoran♂',   'Poison', None,     46,57,40,40,40,50,273),
        (33, 'Nidorino',    'Poison', None,     61,72,57,55,55,65,365),
        (34, 'Nidoking',    'Poison', 'Ground', 81,102,77,85,75,85,505),
        (35, 'Clefairy',    'Fairy',  None,     70,45,48,60,65,35,323),
        (36, 'Clefable',    'Fairy',  None,     95,70,73,95,90,60,483),
        (37, 'Vulpix',      'Fire',   None,     38,41,40,50,65,65,299),
        (38, 'Ninetales',   'Fire',   None,     73,76,75,81,100,100,505),
        (39, 'Jigglypuff',  'Normal', 'Fairy',  115,45,20,45,25,20,270),
        (40, 'Wigglytuff',  'Normal', 'Fairy',  140,70,45,85,50,45,435),
        (41, 'Zubat',       'Poison', 'Flying', 40,45,35,30,40,55,245),
        (42, 'Golbat',      'Poison', 'Flying', 75,80,70,65,75,90,455),
        (43, 'Oddish',      'Grass',  'Poison', 45,50,55,75,65,30,320),
        (44, 'Gloom',       'Grass',  'Poison', 60,65,70,85,75,40,395),
        (45, 'Vileplume',   'Grass',  'Poison', 75,80,85,110,90,50,490),
        (46, 'Paras',       'Bug',    'Grass',  35,70,55,45,55,25,285),
        (47, 'Parasect',    'Bug',    'Grass',  60,95,80,60,80,30,405),
        (48, 'Venonat',     'Bug',    'Poison', 60,55,50,40,55,45,305),
        (49, 'Venomoth',    'Bug',    'Poison', 70,65,60,90,75,90,450),
        (50, 'Diglett',     'Ground', None,     10,55,25,35,45,95,265),
        (51, 'Dugtrio',     'Ground', None,     35,100,50,50,70,120,425),
        (52, 'Meowth',      'Normal', None,     40,45,35,40,40,90,290),
        (53, 'Persian',     'Normal', None,     65,70,60,65,65,115,440),
        (54, 'Psyduck',     'Water',  None,     50,52,48,65,50,55,320),
        (55, 'Golduck',     'Water',  None,     80,82,78,95,80,85,500),
        (56, 'Mankey',      'Fighting',None,    40,80,35,35,45,70,305),
        (57, 'Primeape',    'Fighting',None,    65,105,60,60,70,95,455),
        (58, 'Growlithe',   'Fire',   None,     55,70,45,70,50,60,350),
        (59, 'Arcanine',    'Fire',   None,     90,110,80,100,80,95,555),
        (60, 'Poliwag',     'Water',  None,     40,50,40,40,40,90,300),
        (61, 'Poliwhirl',   'Water',  None,     65,65,65,50,50,90,385),
        (62, 'Poliwrath',   'Water',  'Fighting',90,95,95,70,90,70,510),
        (63, 'Abra',        'Psychic',None,     25,20,15,105,55,90,310),
        (64, 'Kadabra',     'Psychic',None,     40,35,30,120,70,105,400),
        (65, 'Alakazam',    'Psychic',None,     55,50,45,135,95,120,500),
        (66, 'Machop',      'Fighting',None,    70,80,50,35,35,35,305),
        (67, 'Machoke',     'Fighting',None,    80,100,70,50,60,45,405),
        (68, 'Machamp',     'Fighting',None,    90,130,80,65,85,55,505),
        (69, 'Bellsprout',  'Grass',  'Poison', 50,75,35,70,30,40,300),
        (70, 'Weepinbell',  'Grass',  'Poison', 65,90,50,85,45,55,390),
        (71, 'Victreebel',  'Grass',  'Poison', 80,105,65,100,70,70,490),
        (72, 'Tentacool',   'Water',  'Poison', 40,40,35,50,100,70,335),
        (73, 'Tentacruel',  'Water',  'Poison', 80,70,65,80,120,100,515),
        (74, 'Geodude',     'Rock',   'Ground', 40,80,100,30,30,20,300),
        (75, 'Graveler',    'Rock',   'Ground', 55,95,115,45,45,35,390),
        (76, 'Golem',       'Rock',   'Ground', 80,120,130,55,65,45,495),
        (77, 'Ponyta',      'Fire',   None,     50,85,55,65,65,90,410),
        (78, 'Rapidash',    'Fire',   None,     65,100,70,80,80,105,500),
        (79, 'Slowpoke',    'Water',  'Psychic',90,65,65,40,40,15,315),
        (80, 'Slowbro',     'Water',  'Psychic',95,75,110,100,80,30,490),
        (81, 'Magnemite',   'Electric','Steel', 25,35,70,95,55,45,325),
        (82, 'Magneton',    'Electric','Steel', 50,60,95,120,70,70,465),
        (83, 'Farfetchd',   'Normal', 'Flying', 52,90,55,58,62,60,377),
        (84, 'Doduo',       'Normal', 'Flying', 35,85,45,35,35,75,310),
        (85, 'Dodrio',      'Normal', 'Flying', 60,110,70,60,60,110,470),
        (86, 'Seel',        'Water',  None,     65,45,55,45,70,45,325),
        (87, 'Dewgong',     'Water',  'Ice',    90,70,80,70,95,70,475),
        (88, 'Grimer',      'Poison', None,     80,80,50,40,50,25,325),
        (89, 'Muk',         'Poison', None,     105,105,75,65,100,50,500),
        (90, 'Shellder',    'Water',  None,     30,65,100,45,25,40,305),
        (91, 'Cloyster',    'Water',  'Ice',    50,95,180,85,45,70,525),
        (92, 'Gastly',      'Ghost',  'Poison', 30,35,30,100,35,80,310),
        (93, 'Haunter',     'Ghost',  'Poison', 45,50,45,115,55,95,405),
        (94, 'Gengar',      'Ghost',  'Poison', 60,65,60,130,75,110,500),
        (95, 'Onix',        'Rock',   'Ground', 35,45,160,30,45,70,385),
        (96, 'Drowzee',     'Psychic',None,     60,48,45,43,90,42,328),
        (97, 'Hypno',       'Psychic',None,     85,73,70,73,115,67,483),
        (98, 'Krabby',      'Water',  None,     30,105,90,25,25,50,325),
        (99, 'Kingler',     'Water',  None,     55,130,115,50,50,75,475),
        (100,'Voltorb',     'Electric',None,    40,30,50,55,55,100,330),
        (101,'Electrode',   'Electric',None,    60,50,70,80,80,150,490),
        (102,'Exeggcute',   'Grass',  'Psychic',60,40,80,60,45,40,325),
        (103,'Exeggutor',   'Grass',  'Psychic',95,95,85,125,75,55,530),
        (104,'Cubone',      'Ground', None,     50,50,95,40,50,35,320),
        (105,'Marowak',     'Ground', None,     60,80,110,50,80,45,425),
        (106,'Hitmonlee',   'Fighting',None,    50,120,53,35,110,87,455),
        (107,'Hitmonchan',  'Fighting',None,    50,105,79,35,110,76,455),
        (108,'Lickitung',   'Normal', None,     90,55,75,60,75,30,385),
        (109,'Koffing',     'Poison', None,     40,65,95,60,45,35,340),
        (110,'Weezing',     'Poison', None,     65,90,120,85,70,60,490),
        (111,'Rhyhorn',     'Ground', 'Rock',   80,85,95,30,30,25,345),
        (112,'Rhydon',      'Ground', 'Rock',   105,130,120,45,45,40,485),
        (113,'Chansey',     'Normal', None,     250,5,5,35,105,50,450),
        (114,'Tangela',     'Grass',  None,     65,55,115,100,40,60,435),
        (115,'Kangaskhan',  'Normal', None,     105,95,80,40,80,90,490),
        (116,'Horsea',      'Water',  None,     30,40,70,70,25,60,295),
        (117,'Seadra',      'Water',  None,     55,65,95,95,45,85,440),
        (118,'Goldeen',     'Water',  None,     45,67,60,35,50,63,320),
        (119,'Seaking',     'Water',  None,     80,92,65,65,80,68,450),
        (120,'Staryu',      'Water',  None,     30,45,55,70,55,85,340),
        (121,'Starmie',     'Water',  'Psychic',60,75,85,100,85,115,520),
        (122,'Mr. Mime',    'Psychic','Fairy',  40,45,65,100,120,90,460),
        (123,'Scyther',     'Bug',    'Flying', 70,110,80,55,80,105,500),
        (124,'Jynx',        'Ice',    'Psychic',65,50,35,115,95,95,455),
        (125,'Electabuzz',  'Electric',None,    65,83,57,95,85,105,490),
        (126,'Magmar',      'Fire',   None,     65,95,57,100,85,93,495),
        (127,'Pinsir',      'Bug',    None,     65,125,100,55,70,85,500),
        (128,'Tauros',      'Normal', None,     75,100,95,40,70,110,490),
        (129,'Magikarp',    'Water',  None,     20,10,55,15,20,80,200),
        (130,'Gyarados',    'Water',  'Flying', 95,125,79,60,100,81,540),
        (131,'Lapras',      'Water',  'Ice',    130,85,80,85,95,60,535),
        (132,'Ditto',       'Normal', None,     48,48,48,48,48,48,288),
        (133,'Eevee',       'Normal', None,     55,55,50,45,65,55,325),
        (134,'Vaporeon',    'Water',  None,     130,65,60,110,95,65,525),
        (135,'Jolteon',     'Electric',None,    65,65,60,110,95,130,525),
        (136,'Flareon',     'Fire',   None,     65,130,60,95,110,65,525),
        (137,'Porygon',     'Normal', None,     65,60,70,85,75,40,395),
        (138,'Omanyte',     'Rock',   'Water',  35,40,100,90,55,35,355),
        (139,'Omastar',     'Rock',   'Water',  70,60,125,115,70,55,495),
        (140,'Kabuto',      'Rock',   'Water',  30,80,90,55,45,55,355),
        (141,'Kabutops',    'Rock',   'Water',  60,115,105,65,70,80,495),
        (142,'Aerodactyl',  'Rock',   'Flying', 80,105,65,60,75,130,515),
        (143,'Snorlax',     'Normal', None,     160,110,65,65,110,30,540),
        (144,'Articuno',    'Ice',    'Flying', 90,85,100,95,125,85,580),
        (145,'Zapdos',      'Electric','Flying',90,90,85,125,90,100,580),
        (146,'Moltres',     'Fire',   'Flying', 90,100,90,125,85,90,580),
        (147,'Dratini',     'Dragon', None,     41,64,45,50,50,50,300),
        (148,'Dragonair',   'Dragon', None,     61,84,65,70,70,70,420),
        (149,'Dragonite',   'Dragon', 'Flying', 91,134,95,100,100,80,600),
        (150,'Mewtwo',      'Psychic',None,     106,110,90,154,90,130,680),
        (151,'Mew',         'Psychic',None,     100,100,100,100,100,100,600),
    ]

    # ── Tipos ────────────────────────────────────────────────────────────────
    types_data = [
        (1,'Normal'),(2,'Fire'),(3,'Water'),(4,'Electric'),(5,'Grass'),
        (6,'Ice'),(7,'Fighting'),(8,'Poison'),(9,'Ground'),(10,'Flying'),
        (11,'Psychic'),(12,'Bug'),(13,'Rock'),(14,'Ghost'),(15,'Dragon'),
        (16,'Steel'),(17,'Fairy'),
    ]

    # ── Evoluções ────────────────────────────────────────────────────────────
    evolutions_data = [
        (1,  2,  16,   'level'),
        (2,  3,  32,   'level'),
        (4,  5,  16,   'level'),
        (5,  6,  36,   'level'),
        (7,  8,  16,   'level'),
        (8,  9,  36,   'level'),
        (10, 11, 7,    'level'),
        (11, 12, 10,   'level'),
        (13, 14, 7,    'level'),
        (14, 15, 10,   'level'),
        (16, 17, 18,   'level'),
        (17, 18, 36,   'level'),
        (19, 20, 20,   'level'),
        (21, 22, 20,   'level'),
        (23, 24, 22,   'level'),
        (25, 26, None, 'stone'),
        (27, 28, 22,   'level'),
        (29, 30, 16,   'level'),
        (30, 31, None, 'stone'),
        (32, 33, 16,   'level'),
        (33, 34, None, 'stone'),
        (35, 36, None, 'stone'),
        (37, 38, None, 'stone'),
        (39, 40, None, 'stone'),
        (41, 42, 22,   'level'),
        (43, 44, 21,   'level'),
        (44, 45, None, 'stone'),
        (46, 47, 24,   'level'),
        (48, 49, 31,   'level'),
        (50, 51, 26,   'level'),
        (52, 53, 28,   'level'),
        (54, 55, 33,   'level'),
        (56, 57, 28,   'level'),
        (58, 59, None, 'stone'),
        (60, 61, 25,   'level'),
        (61, 62, None, 'stone'),
        (63, 64, 16,   'level'),
        (64, 65, None, 'trade'),
        (66, 67, 28,   'level'),
        (67, 68, None, 'trade'),
        (69, 70, 21,   'level'),
        (70, 71, None, 'stone'),
        (72, 73, 30,   'level'),
        (74, 75, 25,   'level'),
        (75, 76, None, 'trade'),
        (77, 78, 40,   'level'),
        (79, 80, 37,   'level'),
        (81, 82, 30,   'level'),
        (84, 85, 31,   'level'),
        (86, 87, 34,   'level'),
        (88, 89, 38,   'level'),
        (90, 91, None, 'stone'),
        (92, 93, 25,   'level'),
        (93, 94, None, 'trade'),
        (96, 97, 26,   'level'),
        (98, 99, 28,   'level'),
        (100,101,30,   'level'),
        (102,103,None, 'stone'),
        (104,105,28,   'level'),
        (109,110,35,   'level'),
        (111,112,42,   'level'),
        (116,117,32,   'level'),
        (118,119,33,   'level'),
        (120,121,None, 'stone'),
        (129,130,20,   'level'),
        (133,134,None, 'stone'),
        (133,135,None, 'stone'),
        (133,136,None, 'stone'),
        (138,139,40,   'level'),
        (140,141,40,   'level'),
        (147,148,30,   'level'),
        (148,149,55,   'level'),
    ]

    # ── Moves (seleção dos mais icônicos da Gen 1) ───────────────────────────
    moves_data = [
        (1,  'Tackle',        'Normal',   40, 100, 35, 'Physical'),
        (2,  'Growl',         'Normal',   None,100, 40, 'Status'),
        (3,  'Vine Whip',     'Grass',    45, 100, 25, 'Physical'),
        (4,  'Razor Leaf',    'Grass',    55, 95,  25, 'Physical'),
        (5,  'Solar Beam',    'Grass',    120,100, 10, 'Special'),
        (6,  'Scratch',       'Normal',   40, 100, 35, 'Physical'),
        (7,  'Ember',         'Fire',     40, 100, 25, 'Special'),
        (8,  'Flamethrower',  'Fire',     90, 100, 15, 'Special'),
        (9,  'Fire Blast',    'Fire',     110,85,  5,  'Special'),
        (10, 'Water Gun',     'Water',    40, 100, 25, 'Special'),
        (11, 'Surf',          'Water',    90, 100, 15, 'Special'),
        (12, 'Hydro Pump',    'Water',    110,80,  5,  'Special'),
        (13, 'Thunderbolt',   'Electric', 90, 100, 15, 'Special'),
        (14, 'Thunder',       'Electric', 110,70,  10, 'Special'),
        (15, 'Psychic',       'Psychic',  90, 100, 10, 'Special'),
        (16, 'Hyper Beam',    'Normal',   150,90,  5,  'Special'),
        (17, 'Body Slam',     'Normal',   85, 100, 15, 'Physical'),
        (18, 'Earthquake',    'Ground',   100,100, 10, 'Physical'),
        (19, 'Ice Beam',      'Ice',      90, 100, 10, 'Special'),
        (20, 'Blizzard',      'Ice',      110,70,  5,  'Special'),
        (21, 'Slash',         'Normal',   70, 100, 20, 'Physical'),
        (22, 'Dragon Rage',   'Dragon',   None,100, 10, 'Special'),
        (23, 'Night Shade',   'Ghost',    None,100, 15, 'Special'),
        (24, 'Toxic',         'Poison',   None,90,  10, 'Status'),
        (25, 'Swords Dance',  'Normal',   None,None,30, 'Status'),
        (26, 'Amnesia',       'Psychic',  None,None,20, 'Status'),
        (27, 'Recover',       'Normal',   None,None,10, 'Status'),
        (28, 'Softboiled',    'Normal',   None,None,10, 'Status'),
        (29, 'Thunder Wave',  'Electric', None,90,  20, 'Status'),
        (30, 'Mega Punch',    'Normal',   80, 85,  20, 'Physical'),
    ]

    # ── pokemon_moves (subset representativo) ───────────────────────────────
    pokemon_moves_data = [
        (1,  1,  'level-up', 1), (1,  2,  'level-up', 1),
        (1,  3,  'level-up', 7), (1,  4,  'level-up', 27),
        (1,  5,  'level-up', 45),(1,  24, 'tm',       None),
        (4,  6,  'level-up', 1), (4,  7,  'level-up', 9),
        (4,  8,  'level-up', 38),(4,  9,  'tm',       None),
        (7,  1,  'level-up', 1), (7,  10, 'level-up', 7),
        (7,  11, 'tm',       None),(7, 12, 'level-up', 42),
        (25, 1,  'level-up', 1), (25, 29, 'level-up', 1),
        (25, 13, 'level-up', 26),(25, 14, 'tm',       None),
        (6,  8,  'level-up', 46),(6,  9,  'tm',       None),
        (9,  11, 'tm',       None),(9, 12, 'level-up', 52),
        (94, 23, 'level-up', 1), (94, 15, 'tm',       None),
        (65, 15, 'level-up', 36),(65, 26, 'level-up', 47),
        (149,22, 'level-up', 1), (149,16, 'tm',       None),
        (150,15, 'level-up', 1), (150,16, 'tm',       None),
        (143,17, 'level-up', 35),(143,18, 'tm',       None),
        (130,16, 'tm',       None),(130,11,'tm',       None),
    ]

    c.executemany("INSERT OR IGNORE INTO pokemon VALUES (?,?,?,?,?,?,?,?,?,?,?)", pokemon_data)
    c.executemany("INSERT OR IGNORE INTO types VALUES (?,?)", types_data)
    c.executemany("INSERT OR IGNORE INTO evolutions (from_id,to_id,min_level,method) VALUES (?,?,?,?)", evolutions_data)
    c.executemany("INSERT OR IGNORE INTO moves VALUES (?,?,?,?,?,?,?)", moves_data)
    c.executemany("INSERT OR IGNORE INTO pokemon_moves VALUES (?,?,?,?)", pokemon_moves_data)

    conn.commit()
    conn.close()
    logger.info("Pokédex Gen 1 criada/atualizada ✓")

create_sample_database()

In [13]:
# Schema string reusado por várias signatures
DB_SCHEMA = """
Tables and columns:
  pokemon     : id (PK), name TEXT, type1 TEXT, type2 TEXT (nullable), hp INT,
                attack INT, defense INT, sp_atk INT, sp_def INT, speed INT, total INT
  types       : id (PK), name TEXT UNIQUE
  evolutions  : id (PK), from_id INT (FK→pokemon.id), to_id INT (FK→pokemon.id),
                min_level INT (nullable), method TEXT ('level'|'stone'|'trade')
                NOTE: each row is one evolution step; use WITH RECURSIVE to get full chains
  moves       : id (PK), name TEXT, type TEXT, power INT (nullable),
                accuracy INT (nullable), pp INT, category TEXT ('Physical'|'Special'|'Status')
  pokemon_moves: pokemon_id (FK→pokemon.id), move_id (FK→moves.id),
                learn_method TEXT ('level-up'|'tm'|'hm'), learn_level INT (nullable)
"""

def get_live_schema(db_path: str = "pokedex.db") -> str:
    """Retorna schema real lido do SQLite (útil para bancos dinâmicos)."""
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    conn.close()
    return "\n".join(f"-- {name}\n{sql};" for name, sql in rows)

print(get_live_schema())


-- evolutions
CREATE TABLE evolutions (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            from_id      INTEGER NOT NULL,
            to_id        INTEGER NOT NULL,
            min_level    INTEGER,        -- NULL se não evoluir por level
            method       TEXT NOT NULL,  -- 'level', 'stone', 'trade'
            FOREIGN KEY (from_id) REFERENCES pokemon(id),
            FOREIGN KEY (to_id)   REFERENCES pokemon(id)
        );
-- moves
CREATE TABLE moves (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL,
            type     TEXT NOT NULL,
            power    INTEGER,   -- NULL para moves de status
            accuracy INTEGER,
            pp       INTEGER NOT NULL,
            category TEXT NOT NULL  -- 'Physical', 'Special', 'Status'
        );
-- pokemon
CREATE TABLE pokemon (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL,
            type1    TEXT NOT NULL,
            type2    TEXT,           -- 

## 4. Signatures



In [14]:
# Signatures Aprimoradas

class GenerateSQL(dspy.Signature):
    """Translate a natural language question into a valid SQLite query.

    Rules:
    - Use only tables and columns listed in the schema.
    - Return ONLY the SQL statement, no explanation, no markdown fences.
    - Use LIMIT 100 by default unless the question asks for all rows.
    - Use standard SQLite syntax (no CTEs with RECURSIVE unless needed).
    """
    schema:   str = dspy.InputField(desc="Full database schema with table and column definitions")
    question: str = dspy.InputField(desc="Natural language question about the database")
    sql_query: str = dspy.OutputField(desc="Valid SQLite SELECT statement, nothing else")


class RefineSQL(dspy.Signature):
    """Fix a broken SQL query given the original question, the faulty query, and the error message.

    Rules:
    - Keep the same semantic intent as the original question.
    - Fix only what caused the error.
    - Return ONLY the corrected SQL statement.
    """
    schema:    str = dspy.InputField(desc="Database schema")
    question:  str = dspy.InputField(desc="Original natural language question")
    sql_query: str = dspy.InputField(desc="SQL query that produced an error")
    error:     str = dspy.InputField(desc="SQLite error message")
    refined_sql: str = dspy.OutputField(desc="Corrected SQL statement only")


class InterpretResults(dspy.Signature):
    """Interpret SQL query results and produce a concise, friendly natural-language answer.

    - If results are empty, say so clearly.
    - Round floats to 2 decimal places.
    - Be concise — at most 3 sentences.
    """
    question: str  = dspy.InputField(desc="The original question the user asked")
    sql_query: str = dspy.InputField(desc="The SQL query that was executed")
    results:   str = dspy.InputField(desc="Query results as a JSON-encoded list of rows")
    answer:    str = dspy.OutputField(desc="Friendly natural-language answer to the question")


class ValidateSQL(dspy.Signature):
    """Decide if a SQL query is safe and semantically correct before execution.

    Respond with is_valid=True only if:
    - The query uses ONLY SELECT.
    - Read-only SQLite functions like strftime, COUNT, SUM, AVG are perfectly VALID and SAFE.
    - NO destructive commands (INSERT, UPDATE, DELETE, DROP).
    - ONLY answers related to the context of database.
    """
    schema:    str  = dspy.InputField(desc="Database schema")
    sql_query: str  = dspy.InputField(desc="SQL query to evaluate")
    is_valid:  bool = dspy.OutputField(desc="True if safe to execute, False otherwise")
    reason:    str  = dspy.OutputField(desc="Short explanation")


class ClassifyQuestion(dspy.Signature):
    """Classify a database question by complexity.

    Categories:
    - 'simple'  : single-table lookup, no aggregation
    - 'medium'  : aggregation (COUNT/SUM/AVG) or single JOIN
    - 'complex' : multiple JOINs, subqueries, or window functions
    """
    question:   str = dspy.InputField(desc="Natural language question")
    complexity: str = dspy.OutputField(desc="One of: simple | medium | complex")
    reasoning:  str = dspy.OutputField(desc="One sentence explaining the classification")


print("✓ Signatures carregadas")


✓ Signatures carregadas


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "GenerateSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)
/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "RefineSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)
/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "ValidateSQL" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


## 5. Modules

In [15]:
# Modules Aprimorados

class SQLValidator(dspy.Module):
    """Validador determinístico em Python puro. Não usa o LLM."""
    def __init__(self):
        super().__init__()
        # Palavras estritamente proibidas em consultas de leitura
        self.forbidden_keywords = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "REPLACE", "TRUNCATE"]

    def forward(self, schema: str, sql_query: str) -> dspy.Prediction:
        upper_sql = sql_query.upper()

        # Checa se alguma palavra proibida está na string do SQL
        for word in self.forbidden_keywords:
            if f" {word} " in upper_sql or upper_sql.startswith(word):
                return dspy.Prediction(
                    is_valid=False,
                    reason=f"Segurança: A query contém o comando destrutivo '{word}'."
                )

        return dspy.Prediction(is_valid=True, reason="Query de leitura segura.")

class SQLGeneratorSimple(dspy.Module):
    """Gerador simples com ChainOfThought — apenas gera o SQL."""
    def __init__(self):
        super().__init__()
        self.generator = dspy.ChainOfThought(GenerateSQL)

    def forward(self, question: str, schema: str) -> dspy.Prediction:
        pred = self.generator(schema=schema, question=question)
        sql = pred.sql_query.strip().strip('`') # Correção sintática
        return dspy.Prediction(sql_query=sql, strategy="simple")

class SQLGeneratorMultiChain(dspy.Module):
    """Gera M candidatos via ChainOfThought e usa o comparador para escolher o melhor."""
    def __init__(self, M: int = 3):
        super().__init__()
        self.M = M
        # 1. Módulo para GERAR as múltiplas respostas
        self.generator = dspy.ChainOfThought(GenerateSQL)
        # 2. Módulo para COMPARAR as respostas geradas
        self.comparator = dspy.MultiChainComparison(GenerateSQL)

    def forward(self, question: str, schema: str) -> dspy.Prediction:
        # 1. Gera M respostas independentes pedindo config(n=M) para o LLM
        raw_preds = self.generator(question=question, schema=schema, config=dict(n=self.M))

        # 2. Avalia e escolhe a melhor resposta passando as 'completions' obrigatoriamente
        best_pred = self.comparator(completions=raw_preds.completions, question=question, schema=schema)

        sql = best_pred.sql_query.strip().strip('`')
        return dspy.Prediction(sql_query=sql, strategy=f"multi_chain(M={self.M})")

class SQLRefiner(dspy.Module):
    """Refina uma query que falhou com base no erro do banco de dados."""
    def __init__(self):
        super().__init__()
        self.refiner = dspy.ChainOfThought(RefineSQL)

    def forward(self, question: str, schema: str, sql_query: str, error: str) -> dspy.Prediction:
        pred = self.refiner(schema=schema, question=question, sql_query=sql_query, error=error)
        sql = pred.refined_sql.strip().strip('`')
        return dspy.Prediction(sql_query=sql)

class SQLInterpreter(dspy.Module):
    """Converte resultados brutos em linguagem natural."""
    def __init__(self):
        super().__init__()
        self.interpret = dspy.ChainOfThought(InterpretResults)

    def forward(self, question: str, sql_query: str, results) -> str:
        results_json = json.dumps(results[:50])
        pred = self.interpret(question=question, sql_query=sql_query, results=results_json)
        return pred.answer

print("✓ Modules corrigidos e carregados")


✓ Modules corrigidos e carregados


## 6. Pipeline Principal do Agente

In [16]:
# Pipeline completo com segurança rigorosa

class QueryCache:
    def __init__(self):
        self._store: dict = {}
    def key(self, question: str) -> str:
        return hashlib.md5(question.strip().lower().encode()).hexdigest()
    def get(self, question: str):
        return self._store.get(self.key(question))
    def set(self, question: str, value):
        self._store[self.key(question)] = value
    def clear(self):
        self._store.clear()

class SQLAgent(dspy.Module):
    """
    Agente principal com execução segura.
    Fluxo: Classificar -> Gerar -> Validar -> (Se válido) Executar -> (Se erro) Refinar -> Validar -> Executar
    """
    def __init__(self, db_path: str = "pokedex.db", use_cache: bool = True):
        super().__init__()
        self.db_path    = db_path
        self.schema     = DB_SCHEMA
        self.use_cache  = use_cache

        # Módulos instanciados
        self.classifier    = dspy.Predict(ClassifyQuestion)
        self.validator     = SQLValidator()
        self.gen_simple    = SQLGeneratorSimple()
        self.gen_multi     = SQLGeneratorMultiChain(M=3)
        self.refiner       = SQLRefiner()
        self.interpreter   = SQLInterpreter()

        self.cache   = QueryCache()
        self.history: list = []

    def _get_conn(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def forward(self, question: str, interpret: bool = True) -> dict:
        start_time = time.time()

        if self.use_cache:
            cached = self.cache.get(question)
            if cached:
                logger.info("Cache hit para: %s", question[:60])
                return {**cached, "from_cache": True}

        entry = {
            "question":    question,
            "timestamp":   datetime.now().isoformat(),
            "from_cache":  False,
            "error":       None
        }

        conn = self._get_conn()
        try:
            # Classificação
            cls = self.classifier(question=question)
            complexity = cls.complexity.strip().lower()
            entry["complexity"] = complexity

            # Geração Pura (Roteamento único)
            # Devido à limitação do Ollama local não suportar o parâmetro 'n',
            # roteamos tudo para o gerador simples e confiamos no pipeline de Refinamento.
            gen_pred = self.gen_simple(question=question, schema=self.schema)

            sql_candidate = gen_pred.sql_query
            strategy = gen_pred.strategy

            # Função auxiliar de tentativa de execução
            def try_execute(sql_to_run):
                val = self.validator(schema=self.schema, sql_query=sql_to_run)
                entry["validation"] = {"is_valid": val.is_valid, "reason": val.reason}

                if not val.is_valid:
                    raise Exception(f"Bloqueado pelo validador: {val.reason}")

                # Execução
                return conn.execute(sql_to_run).fetchall()

            # Primeira tentativa
            try:
                raw_results = try_execute(sql_candidate)
                entry["sql_query"] = sql_candidate
                entry["strategy"]  = strategy
            except Exception as e:
                # Pipeline de Refinamento
                error_msg = str(e)
                logger.info(f"Tentativa inicial falhou, iniciando refinamento. Erro: {error_msg}")
                refine_pred = self.refiner(question=question, schema=self.schema, sql_query=sql_candidate, error=error_msg)
                sql_refined = refine_pred.sql_query

                try:
                    raw_results = try_execute(sql_refined)
                    entry["sql_query"] = sql_refined
                    entry["strategy"]  = strategy + "+refined"
                except Exception as e_refined:
                    entry["sql_query"] = sql_refined
                    entry["strategy"]  = strategy + "+refined_failed"
                    raise Exception(f"Falha mesmo após refinamento: {e_refined}")

            # Serialização e Interpretação
            results = [tuple(r) for r in raw_results]
            entry["results"] = results
            entry["result_count"] = len(results)

            if interpret:
                entry["answer"] = self.interpreter(
                    question=question,
                    sql_query=entry["sql_query"],
                    results=results,
                )
            else:
                entry["answer"] = None

        except Exception as final_error:
            entry["error"]  = str(final_error)
            entry["answer"] = f"Não foi possível resolver a consulta: {final_error}"
            logger.error("Erro no fluxo: %s", final_error)
        finally:
            conn.close()

        entry["elapsed_ms"] = round((time.time() - start_time) * 1000)

        self.history.append(entry)
        if self.use_cache and not entry.get("error"):
            self.cache.set(question, entry)

        return entry

agent = SQLAgent(use_cache=True)
print("✓ SQLAgent inicializado com pipeline seguro")

✓ SQLAgent inicializado com pipeline seguro


/usr/local/lib/python3.12/dist-packages/dspy/signatures/signature.py:185: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  cls = super().__new__(mcs, signature_name, bases, namespace, **kwargs)


## 7. Exemplos de Treinamento (Few-Shot)

In [17]:
def create_training_examples() -> list:
    examples = [
        # Simple
        dspy.Example(
            schema=DB_SCHEMA,
            question="How many Fire-type Pokémon are there in Gen 1?",
            sql_query="SELECT COUNT(*) AS total FROM pokemon WHERE type1 = 'Fire' OR type2 = 'Fire'",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="What is the highest base stat total in Gen 1?",
            sql_query="SELECT name, total FROM pokemon ORDER BY total DESC LIMIT 1",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="List all Psychic-type Pokémon",
            sql_query="SELECT id, name FROM pokemon WHERE type1 = 'Psychic' OR type2 = 'Psychic' ORDER BY id",
        ).with_inputs("schema", "question"),

        # Medium
        dspy.Example(
            schema=DB_SCHEMA,
            question="What is the average speed of Water-type Pokémon?",
            sql_query="SELECT ROUND(AVG(speed), 2) AS avg_speed FROM pokemon WHERE type1 = 'Water' OR type2 = 'Water'",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Which Pokémon evolve by stone?",
            sql_query="SELECT p.name AS from_pokemon, p2.name AS to_pokemon FROM evolutions e JOIN pokemon p ON e.from_id = p.id JOIN pokemon p2 ON e.to_id = p2.id WHERE e.method = 'stone'",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Show average HP grouped by primary type",
            sql_query="SELECT type1, ROUND(AVG(hp), 2) AS avg_hp FROM pokemon GROUP BY type1 ORDER BY avg_hp DESC",
        ).with_inputs("schema", "question"),

        # Complex
        dspy.Example(
            schema=DB_SCHEMA,
            question="Which Pokémon have more than 100 base attack and are not pure Normal type?",
            sql_query="SELECT name, attack, type1, type2 FROM pokemon WHERE attack > 100 AND type1 != 'Normal' ORDER BY attack DESC",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="List the full evolution chain of Charmander",
            sql_query="SELECT p1.name AS stage1, p2.name AS stage2, p3.name AS stage3 FROM evolutions e1 JOIN evolutions e2 ON e1.to_id = e2.from_id JOIN pokemon p1 ON e1.from_id = p1.id JOIN pokemon p2 ON e1.to_id = p2.id JOIN pokemon p3 ON e2.to_id = p3.id WHERE p1.name = 'Charmander'",
        ).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Which Special moves have power above 100?",
            sql_query="SELECT name, type, power FROM moves WHERE category = 'Special' AND power > 100 ORDER BY power DESC",
        ).with_inputs("schema", "question"),

dspy.Example(
    schema=DB_SCHEMA,
    question="What is the full evolution line of Charmander?",
    sql_query="""
WITH RECURSIVE evo_chain(id, name, step) AS (
    SELECT p.id, p.name, 0
    FROM pokemon p
    WHERE p.name = 'Charmander'
    UNION ALL
    SELECT p.id, p.name, ec.step + 1
    FROM evo_chain ec
    JOIN evolutions e ON ec.id = e.from_id
    JOIN pokemon p ON e.to_id = p.id
)
SELECT name, step FROM evo_chain ORDER BY step
""",
).with_inputs("schema", "question"),

        dspy.Example(
            schema=DB_SCHEMA,
            question="Which Pokémon can learn Surf (move_id=11)?",
            sql_query="SELECT p.name FROM pokemon p JOIN pokemon_moves pm ON p.id = pm.pokemon_id WHERE pm.move_id = 11 ORDER BY p.id",
        ).with_inputs("schema", "question"),
    ]
    return examples

    TRAIN_EXAMPLES = create_training_examples()
    print("✓ Exemplos de treinamento carregados")

## 8. Métrica de Avaliação

In [20]:
def sql_accuracy_metric(example: dspy.Example, pred: dspy.Prediction, trace=None) -> float:
    """
    Métrica composta para avaliar qualidade do SQL gerado.

    Pontuação:
    - +0.5 se a query executa sem erro
    - +0.3 se o número de colunas retornadas bate com o esperado
    - +0.2 se as palavras-chave essenciais do SQL esperado estão presentes
    """
    score = 0.0
    predicted_sql = getattr(pred, "sql_query", "").strip()
    expected_sql  = example.sql_query.strip()

    if not predicted_sql:
        return 0.0

    # Tenta executar
    try:
        conn = sqlite3.connect("pokedex.db")
        rows = conn.execute(predicted_sql).fetchall()
        conn.close()
        score += 0.5

        # Verifica número de colunas (compara com expected se disponível)
        try:
            conn2 = sqlite3.connect("pokedex.db")
            exp_rows = conn2.execute(expected_sql).fetchall()
            conn2.close()
            if rows and exp_rows:
                if len(rows[0]) == len(exp_rows[0]):
                    score += 0.3
            elif not rows and not exp_rows:
                score += 0.3  # ambos vazios = correto
        except Exception:
            pass

    except Exception:
        pass

    # Verifica keywords essenciais (FROM, WHERE, GROUP BY, etc.)
    pred_upper = predicted_sql.upper()
    exp_upper  = expected_sql.upper()
    keywords = ["FROM", "WHERE", "GROUP BY", "ORDER BY", "JOIN", "HAVING",
                "COUNT", "SUM", "AVG", "MAX", "MIN"]
    exp_kw  = [kw for kw in keywords if kw in exp_upper]
    pred_kw = [kw for kw in exp_kw if kw in pred_upper]

    if exp_kw:
        score += 0.2 * (len(pred_kw) / len(exp_kw))
    else:
        score += 0.2  # sem keywords obrigatórias = bônus

    return round(score, 3)


import dspy

class _FakePred:
    sql_query = "SELECT COUNT(*) AS total FROM pokemon WHERE type1 = 'Fire' OR type2 = 'Fire'"

# Criamos um "Gabarito" (Mock) temporário apenas para este teste
mock_example = dspy.Example(
    question="Quantos pokémons do tipo fogo existem?",
    sql_query="SELECT COUNT(*) FROM pokemon WHERE type1 = 'Fire' OR type2 = 'Fire'"
).with_inputs("question")

# O teste agora sobrevive independentemente de outras células terem sido executadas ou não
score = sql_accuracy_metric(mock_example, _FakePred())
print(f"Teste da métrica (query correta): {score:.3f}  (esperado ≈ 1.0)")


Teste da métrica (query correta): 1.000  (esperado ≈ 1.0)


## 9. Otimizadores

### Estratégia de otimização
| Otimizador | Quando usar |
|---|---|
| `BootstrapFewShot` | Rápido, sem custo extra de LLM, injeta exemplos validados nos prompts |
| `BootstrapFewShotWithRandomSearch` | Quando há mais tempo e se quer maximizar a métrica com busca aleatória |
| `MIPROv2` | Otimiza *instruções* além dos exemplos — ideal para produção, mais caro |

> **OBS:** Otimizadores do DSPy requerem que o LM seja chamado várias vezes.
> Para Ollama local com modelo pequeno, `BootstrapFewShot` é o mais prático.


In [21]:
from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch

# Otimizador 1: BootstrapFewShot (mais rápido)

def optimize_with_bootstrap(
    module_to_optimize: dspy.Module,
    trainset: list,
    metric,
    max_bootstrapped_demos: int = 4,
    max_labeled_demos: int = 4,
) -> dspy.Module:
    """
    Usa BootstrapFewShot para selecionar os melhores exemplos de few-shot
    automaticamente a partir do trainset.
    """
    logger.info("Iniciando BootstrapFewShot...")
    teleprompter = BootstrapFewShot(
        metric=metric,
        max_bootstrapped_demos=max_bootstrapped_demos,
        max_labeled_demos=max_labeled_demos,
    )
    optimized = teleprompter.compile(
        module_to_optimize,
        trainset=trainset,
    )
    logger.info("BootstrapFewShot concluído ✓")
    return optimized


def optimize_with_random_search(
    module_to_optimize: dspy.Module,
    trainset: list,
    metric,
    num_candidate_programs: int = 8,
    max_bootstrapped_demos: int = 4,
) -> dspy.Module:
    """
    Variante com busca aleatória — testa N configurações e retorna a melhor.
    Mais pesado, mas geralmente produz resultados superiores.
    """
    logger.info("Iniciando BootstrapFewShotWithRandomSearch (num_candidates=%d)...",
                num_candidate_programs)
    teleprompter = BootstrapFewShotWithRandomSearch(
        metric=metric,
        max_bootstrapped_demos=max_bootstrapped_demos,
        num_candidate_programs=num_candidate_programs,
    )
    optimized = teleprompter.compile(
        module_to_optimize,
        trainset=trainset,
    )
    logger.info("RandomSearch concluído ✓")
    return optimized


print("✓ Funções de otimização definidas")


✓ Funções de otimização definidas


In [22]:
# Otimizador 2: MIPROv2 (otimiza instruções + exemplos)

try:
    from dspy.teleprompt import MIPROv2

    def optimize_with_mipro(
        module_to_optimize: dspy.Module,
        trainset: list,
        metric,
        num_trials: int = 10,
        minibatch_size: int = 5,
    ) -> dspy.Module:
        """
        MIPROv2 otimiza tanto as *instruções* das Signatures quanto os exemplos.
        É o otimizador mais poderoso do DSPy para produção.

        Parâmetros recomendados para Ollama local:
        - num_trials=10 (suficiente para modelos menores)
        - minibatch_size=5 (evita sobrecarregar o Ollama)
        """
        logger.info("Iniciando MIPROv2 (trials=%d)...", num_trials)
        teleprompter = MIPROv2(
            metric=metric,
            num_trials=num_trials,
            minibatch_size=minibatch_size,
            verbose=True,
        )
        optimized = teleprompter.compile(
            module_to_optimize,
            trainset=trainset,
            requires_permission_to_run=False,
        )
        logger.info("MIPROv2 concluído ✓")
        return optimized

    print("✓ MIPROv2 disponível")

except ImportError:
    print("⚠ MIPROv2 não disponível nesta versão do dspy-ai — use BootstrapFewShot")
    def optimize_with_mipro(*args, **kwargs):
        raise NotImplementedError("MIPROv2 não disponível. Atualize: pip install -U dspy-ai")


✓ MIPROv2 disponível


## 10. Executando a Otimização

In [24]:
if 'TRAIN_EXAMPLES' not in locals() and 'TRAIN_EXAMPLES' not in globals():
    print("⚠ TRAIN_EXAMPLES não encontrado na memória. Recarregando...")
    TRAIN_EXAMPLES = create_training_examples()


base_module = SQLGeneratorSimple()

# Usamos a divisão focada no Pokedex (3 para treino, resto para validação)
split = 3
train_set = TRAIN_EXAMPLES[:split]
val_set   = TRAIN_EXAMPLES[split:]

print(f"Train: {len(train_set)} exemplos | Val: {len(val_set)} exemplos")


print("\nIniciando otimização. Isso exigirá processamento da CPU...")

optimized_module = optimize_with_bootstrap(
    module_to_optimize=base_module,
    trainset=train_set,
    metric=sql_accuracy_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)


print("\n✓ Módulo otimizado pronto para uso no SQLAgent")

⚠ TRAIN_EXAMPLES não encontrado na memória. Recarregando...
Train: 3 exemplos | Val: 8 exemplos

Iniciando otimização. Isso exigirá processamento da CPU...


100%|██████████| 3/3 [01:39<00:00, 33.02s/it]

Bootstrapped 3 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.

✓ Módulo otimizado pronto para uso no SQLAgent


## 11. Avaliação do Módulo Otimizado

In [25]:
def evaluate_module(module: dspy.Module, examples: list, metric) -> dict:
    """Avalia um módulo em um conjunto de exemplos e retorna métricas agregadas."""
    scores = []
    details = []

    for ex in examples:
        try:
            # Chama o módulo base corretamente, respeitando sua arquitetura e assinatura
            pred = module(schema=ex.schema, question=ex.question)
        except Exception as e:
            pred = dspy.Prediction(sql_query="")

        score = metric(ex, pred)
        scores.append(score)
        details.append({
            "question": ex.question[:55] + "…",
            "score":    score,
            "sql":      getattr(pred, "sql_query", "")[:60] + "…",
        })

    return {
        "mean_score": round(sum(scores) / len(scores), 3) if scores else 0.0,
        "max_score":  round(max(scores), 3) if scores else 0.0,
        "min_score":  round(min(scores), 3) if scores else 0.0,
        "details":    details,
    }

print("=== Avaliação no conjunto de validação ===")
# Agora base_module ou optimized_module serão avaliados da maneira certa
results_eval = evaluate_module(optimized_module, val_set, sql_accuracy_metric)
print(f"  Média:  {results_eval['mean_score']}")
print(f"  Máximo: {results_eval['max_score']}")
print(f"  Mínimo: {results_eval['min_score']}")
print()
print(tabulate(
    [(d["question"], d["score"]) for d in results_eval["details"]],
    headers=["Questão", "Score"],
    tablefmt="rounded_outline"
))

=== Avaliação no conjunto de validação ===
  Média:  0.706
  Máximo: 1.0
  Mínimo: 0.2

╭──────────────────────────────────────────────────────────┬─────────╮
│ Questão                                                  │   Score │
├──────────────────────────────────────────────────────────┼─────────┤
│ What is the average speed of Water-type Pokémon?…        │   1     │
│ Which Pokémon evolve by stone?…                          │   0.633 │
│ Show average HP grouped by primary type…                 │   0.95  │
│ Which Pokémon have more than 100 base attack and are no… │   0.633 │
│ List the full evolution chain of Charmander…             │   0.2   │
│ Which Special moves have power above 100?…               │   0.633 │
│ What is the full evolution line of Charmander?…          │   0.65  │
│ Which Pokémon can learn Surf (move_id=11)?…              │   0.95  │
╰──────────────────────────────────────────────────────────┴─────────╯


## 12. Salvar e Carregar Módulo Otimizado

In [26]:
# Salva pesos otimizados (few-shot demos + instruções)
optimized_module.save("optimized_sql_generator.json")
print("✓ Módulo salvo em optimized_sql_generator.json")

# Para recarregar:
loaded = SQLGeneratorSimple()
loaded.load("optimized_sql_generator.json")


✓ Módulo salvo em optimized_sql_generator.json


## 13. Rodando o Agente Completo

In [27]:
def print_result(entry: dict) -> None:
    """Formata e imprime um resultado do agente."""
    status = "✓" if not entry.get("error") else "✗"
    cache  = " [cache]" if entry.get("from_cache") else ""
    print(f"\n{'='*60}")
    print(f"{status} [{entry.get('complexity','?').upper()}] {entry['question']}{cache}")
    print(f"   SQL      : {entry.get('sql_query','N/A')[:80]}")
    print(f"   Estratégia: {entry.get('strategy','N/A')}")
    if entry.get("error"):
        print(f"   Erro     : {entry['error']}")
    elif entry.get("results") is not None:
        print(f"   Linhas   : {entry.get('result_count', len(entry['results']))}")
    if entry.get("answer"):
        print(f"   Resposta : {entry['answer']}")
    print(f"   Tempo    : {entry.get('elapsed_ms', '?')} ms")


# Conjunto de testes cobrindo diferentes complexidades
test_questions = [
    # Simple
    "How many Pokémon have a base stat total above 500?",
    "What is the average HP of all Gen 1 Pokémon?",
    # Medium
    "Show me all Pokémon with more than 100 base attack",
    "Which Pokémon evolve by trade?",
    "What is the average speed grouped by primary type?",
    # Complex
    "List all Pokémon that have a dual type and their total base stats",
    "Which type combination has the highest average base stat total?",
]

import subprocess
import time
import requests

def garantir_servidor_ollama():
    """Testa se o Ollama está rodando. Se não estiver, força a inicialização."""
    try:
        # Tenta contato com a API
        requests.get("http://localhost:11434/", timeout=2)
        print("✓ Servidor Ollama já está online.")
    except requests.exceptions.ConnectionError:
        print("⚠ Servidor Ollama offline. Reerguendo processo...")
        # Inicia o servidor
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        time.sleep(3) # Aguarda 3 segundos para o serviço respirar
        print("✓ Servidor Ollama reiniciado.")

# Garante que o motor do Ollama está ligado
garantir_servidor_ollama()

# Re-instancia o agente -> força o python a puxar as classes corrigidas da memória recente.
agent = SQLAgent(use_cache=True)
agent.cache.clear() # Limpa resquícios de falhas anteriores

# Roda as perguntas
print("\nIniciando testes...")
for q in test_questions:
    result = agent(q, interpret=True)
    print_result(result)

✓ Servidor Ollama já está online.

Iniciando testes...

✓ [COMPLEX] How many Pokémon have a base stat total above 500?
   SQL      : SELECT COUNT(*) FROM pokemon WHERE total > 500
   Estratégia: simple
   Linhas   : 1
   Resposta : There are 26 Pokémon with a base stat total above 500.
   Tempo    : 4995 ms

✓ [MEDIUM] What is the average HP of all Gen 1 Pokémon?
   SQL      : SELECT AVG(hp) AS average_hp
FROM pokemon
WHERE total BETWEEN 300 AND 600
   Estratégia: simple
   Linhas   : 1
   Resposta : The average HP of all Gen 1 Pokémon with a total stat between 300 and 600 is approximately 67.08.
   Tempo    : 7388 ms

✓ [MEDIUM] Show me all Pokémon with more than 100 base attack
   SQL      : SELECT * FROM pokemon WHERE attack > 100
   Estratégia: simple
   Linhas   : 22
   Resposta : There are 24 Pokémon with more than 100 base attack in the database.
   Tempo    : 5644 ms

✓ [MEDIUM] Which Pokémon evolve by trade?
   SQL      : SELECT DISTINCT p.name FROM pokemon AS p
JOIN evolution

## 14. Demonstração de Cache

In [28]:
print("=== Teste de Cache ===")
q = "How many employees work in Engineering?"

print("1ª chamada (sem cache):")
r1 = agent.forward(q)
print(f"  Tempo: {r1['elapsed_ms']} ms | From cache: {r1['from_cache']}")

print("\n2ª chamada (deve vir do cache):")
r2 = agent.forward(q)
print(f"  Tempo: {r2['elapsed_ms']} ms | From cache: {r2['from_cache']}")


2026/05/07 13:26:03 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.


=== Teste de Cache ===
1ª chamada (sem cache):


ERROR:dspy_agent:Erro no fluxo: Falha mesmo após refinamento: no such table: departments
2026/05/07 13:26:10 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.


  Tempo: 7128 ms | From cache: False

2ª chamada (deve vir do cache):


ERROR:dspy_agent:Erro no fluxo: Falha mesmo após refinamento: no such table: employee


  Tempo: 6915 ms | From cache: False


## 15. Histórico de Execução

In [29]:
def show_history(agent) -> None:
    """Exibe o histórico de queries em formato tabular com proteção de chaves."""
    rows = []
    for i, entry in enumerate(agent.history, 1):
        # Usa .get() com valor padrão para evitar KeyError se a memória tiver "lixo"
        question = entry.get("question", "Questão não registrada")
        rows.append([
            i,
            question[:45] + "…",
            entry.get("complexity", "?"),
            entry.get("strategy", "?"),
            "✓" if not entry.get("error") else "✗",
            f"{entry.get('elapsed_ms','?')} ms",
        ])

    print(tabulate(
        rows,
        headers=["#", "Questão", "Complexidade", "Estratégia", "Status", "Tempo"],
        tablefmt="rounded_outline"
    ))

show_history(agent)


╭─────┬────────────────────────────────────────────────┬────────────────┬───────────────────────┬──────────┬──────────╮
│   # │ Questão                                        │ Complexidade   │ Estratégia            │ Status   │ Tempo    │
├─────┼────────────────────────────────────────────────┼────────────────┼───────────────────────┼──────────┼──────────┤
│   1 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   2 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   3 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   4 │ How many Pokémon have a base stat total above… │ complex        │ simple                │ ✓        │ 4995 ms  │
│   5 │ Questão não registrada…                        │ ?              │ ?                     │ ✓        │ ? ms     │
│   6 │ Questão não registrada…         

## 16. Interface Interativa (REPL)

In [30]:
def interactive_agent(agent: SQLAgent) -> None:
    """
    Loop interativo para consultas em linguagem natural.
    Digite 'sair' para encerrar, 'historico' para ver o log, 'cache clear' para limpar.
    """
    print("\n SQL Agent interativo — digite sua pergunta em inglês")
    print("   Comandos: 'sair' | 'historico' | 'cache clear'\n")

    while True:
        try:
            question = input("Você: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nEncerrando agente.")
            break

        if not question:
            continue
        if question.lower() in ("sair", "exit", "quit"):
            print("Até logo!")
            break
        if question.lower() == "historico":
            show_history(agent)
            continue
        if question.lower() == "cache clear":
            agent.cache.clear()
            print("Cache limpo.")
            continue

        result = agent.forward(question, interpret=True)
        print_result(result)

# Uso interativo:
interactive_agent(agent)
print("ℹ Descomente a última linha para ativar o modo interativo.")



 SQL Agent interativo — digite sua pergunta em inglês
   Comandos: 'sair' | 'historico' | 'cache clear'

Você: quais as evoluções do charmander?


2026/05/07 13:26:35 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.



✓ [MEDIUM] quais as evoluções do charmander?
   SQL      : WITH RECURSIVE evolution_chain AS (
    SELECT e.to_id AS id
    FROM evolutions
   Estratégia: simple
   Linhas   : 2
   Resposta : The evolution chain of Charmander is Charmeleon and Charizard.
   Tempo    : 10029 ms
Você: qual o tipo do charmander?


2026/05/07 13:28:29 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.



✓ [MEDIUM] qual o tipo do charmander?
   SQL      : SELECT type1 FROM pokemon WHERE name = 'Charmander'
   Estratégia: simple
   Linhas   : 1
   Resposta : Charmander is a Fire-type Pokémon.
   Tempo    : 5844 ms
Você: quais os movimentos do charmander?


2026/05/07 13:28:49 WARNING dspy.primitives.module: Calling module.forward(...) on SQLAgent directly is discouraged. Please use module(...) instead.



✓ [MEDIUM] quais os movimentos do charmander?
   SQL      : SELECT DISTINCT m.name FROM pokemon p JOIN pokemon_moves pm ON p.id = pm.pokemon
   Estratégia: simple
   Linhas   : 4
   Resposta : Charmander can perform various attacks including Scratch, Ember, Flamethrower, and Fire Blast.
   Tempo    : 7995 ms

Encerrando agente.
ℹ Descomente a última linha para ativar o modo interativo.


## 17. Resumo das Melhorias Implementadas

| Área | O que foi feito |
|---|---|
| **Signatures** | 5 signatures tipadas: `GenerateSQL`, `RefineSQL`, `InterpretResults`, `ValidateSQL`, `ClassifyQuestion` |
| **Modules** | `SQLValidator`, `SQLGeneratorSimple` (CoT), `SQLGeneratorMultiChain` (MultiChainComparison, M=3), `SQLInterpreter` |
| **MultiChain** | Sim, funciona — gera 3 candidatos e escolhe o melhor; ideal para queries medium/complex |
| **Otimizadores** | `BootstrapFewShot`, `BootstrapFewShotWithRandomSearch`, `MIPROv2` (quando disponível) |
| **Banco de dados** | Expandido para 10 funcionários, 4 departamentos, 6 projetos |
| **Cache** | MD5 hash da pergunta → evita reprocessamento |
| **Histórico** | Log completo com timestamp, complexidade, estratégia, tempo e status |
| **Classificação automática** | Cada query é classificada em *simple/medium/complex* e roteada para o módulo adequado |
| **Avaliação** | Métrica composta (execução + colunas + keywords) + função `evaluate_module` |
| **Persistência** | `module.save()` / `module.load()` para reutilizar o módulo otimizado |
| **REPL interativo** | Loop `interactive_agent()` com comandos de histórico e limpeza de cache |

### Sobre MultiChainComparison
`dspy.MultiChainComparison` funciona gerando **M chains of thought independentes** para o mesmo signature e então aplicando um modelo de comparação para escolher a resposta mais consistente. Isso é análogo a *self-consistency* em chain-of-thought prompting — reduz variância e aumenta confiabilidade especialmente em queries SQL complexas com JOINs e subqueries.
